# FashionGraph — Phase 5, Track B: LP-FT encoder fine-tune (Colab)

Earlier attempts: (1) frozen linear reprojection can't add information; (2) full SupCon encoder
fine-tune **distorts** the pretrained features — val designer top-1 collapsed toward 0 every
epoch (Kumar et al. 2022, *Fine-Tuning can Distort Pretrained Features*, hurts OOD / held-out
collections). This notebook uses the textbook fix, **LP-FT**:

1. **Linear probe** — train a designer/concept head on the *frozen* features.
2. **Fine-tune** — unfreeze the last few blocks at a *tiny* LR, plus an **L2 feature-distillation
   anchor** `\u03bb·||f - f_base||²` that keeps embeddings near the pretrained ones so they can't
   collapse, with light augmentation.

The KG enters as the supervision target: `labels` = designer classification (CE); `concepts` =
multi-label prediction of the designer's KG concepts (BCE). Eval is unchanged: kNN designer
top-k on the honest by-collection test split (head discarded). Reuses the tested repo functions.

**Before you run**, on Google Drive:
```
MyDrive/FashionGraphData/vogue_runway/...      (runway images + .json sidecars)
MyDrive/FashionGraphData/fashion_kg.sqlite     (the built knowledge graph)
```
Then Runtime → GPU (T4) → Run all. Runway imagery is research/thesis-use only.

## 1. GPU check

In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime > Change runtime type > GPU'

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Config — set these to match your setup

In [ ]:
# --- where your data lives on Drive ---
DRIVE_DATA_DIR = '/content/drive/MyDrive/FashionGraphData'
OUTPUT_DIR     = '/content/drive/MyDrive/FashionGraphData/phase5_out'

# --- how to get the repo code ---
REPO_URL  = 'https://github.com/<YOUR_USER>/fashiongraph.git'   # <-- EDIT ME (or use zip below)
REPO_DIR  = '/content/fashiongraph'
USE_DRIVE_ZIP = False
REPO_ZIP  = f'{DRIVE_DATA_DIR}/fashiongraph.zip'

# --- model ---
MODEL_NAME  = 'Marqo/marqo-fashionSigLIP'
SUPERVISION = 'both'         # 'concepts' | 'labels' | 'both'

# --- LP-FT hyper-parameters ---
UNFREEZE_BLOCKS = 2          # keep small — the anchor + tiny LR do the heavy lifting
PROBE_EPOCHS    = 150        # phase 1: head on frozen features
PROBE_LR        = 1e-3
FT_EPOCHS       = 12         # phase 2: encoder fine-tune
FT_LR           = 1e-6       # TINY — distortion killed the last run at 2e-5
ANCHOR_LAMBDA   = 10.0       # strength of the ||f - f_base||^2 anti-collapse anchor
BATCH           = 64
WEIGHT_DECAY    = 1e-4
PATIENCE        = 5          # early-stop on val designer top-1 (floor = base)
TEST_FRAC = 0.2
VAL_FRAC  = 0.2
SEED      = 42
IMG_SIZE  = 256

## 4. Get the repo code + install deps

In [ ]:
import os, sys, subprocess
if USE_DRIVE_ZIP:
    os.makedirs(REPO_DIR, exist_ok=True)
    subprocess.run(['unzip', '-oq', REPO_ZIP, '-d', '/content'], check=True)
else:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=False)
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print('repo at', REPO_DIR)

In [ ]:
!pip -q install open_clip_torch pydantic-settings 2>/dev/null
print('deps ok')

## 5. Load the model (+ train augmentation) and the runway images

In [ ]:
import json, copy, logging
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import open_clip
from PIL import Image

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

hub = f'hf-hub:{MODEL_NAME}'
model, preprocess_train, preprocess = open_clip.create_model_and_transforms(hub)
model = model.to(device)
INIT_STATE = copy.deepcopy(model.state_dict())
print('loaded', MODEL_NAME)

In [ ]:
root = Path(DRIVE_DATA_DIR) / 'vogue_runway'
jsons = sorted(root.rglob('*.json'))
assert jsons, f'No runway JSON under {root} — check DRIVE_DATA_DIR.'
pil_images, metas = [], []
for jp in jsons:
    try:
        info = json.loads(jp.read_text(encoding='utf-8'))
    except Exception:
        continue
    img_path = jp.with_suffix('.png')
    if not img_path.exists():
        rec = info.get('image_path')
        cand = root.parent.parent / rec if rec else None
        if cand and cand.exists():
            img_path = cand
        else:
            continue
    try:
        im = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    except Exception:
        continue
    pil_images.append(im)
    metas.append({'source': 'runway', 'designer': info.get('designer', ''),
                  'show': info.get('show', ''), 'season': info.get('season', ''),
                  'type': info.get('type', ''),
                  'title': f"{info.get('designer','')} \u2014 {info.get('show','')}"})
print(f'loaded {len(pil_images)} runway images')

## 6. Supervision, splits, base embeddings (reuses tested repo code)

In [ ]:
from fg.vision.index import VisualIndex
from fg.kg.store import KnowledgeGraph
from fg.training.alignment_pairs import load_supervision, build_concept_vocab
from fg.training.train_alignment import split_groups_three, designer_topk, bootstrap_ci

kg = KnowledgeGraph(str(Path(DRIVE_DATA_DIR) / 'fashion_kg.sqlite'))

@torch.no_grad()
def embed_all(net, bs=64):
    net.eval()
    outs = []
    for i in range(0, len(pil_images), bs):
        x = torch.stack([preprocess(im) for im in pil_images[i:i+bs]]).to(device)
        f = net.encode_image(x)
        f = f / f.norm(dim=-1, keepdim=True).clamp(min=1e-8)
        outs.append(f.float().cpu().numpy())
    return np.concatenate(outs, 0)

base_emb = embed_all(model)
index = VisualIndex(base_emb, metas)
records = load_supervision(index, kg)
designers = np.array([r.designer for r in records])
fit_rows, val_rows, test_rows = split_groups_three(records, TEST_FRAC, VAL_FRAC, SEED)
ref_rows = np.concatenate([fit_rows, val_rows])
print(f'split: fit={len(fit_rows)} val={len(val_rows)} test={len(test_rows)} (by collection)')
base = designer_topk(base_emb, designers, ref_rows, test_rows)
print('BASE test top1=%.3f top3=%.3f top5=%.3f' % (base['top1'], base['top3'], base['top5']))

## 7. LP-FT: linear probe, then anchored encoder fine-tune

In [ ]:
def visual_blocks(net):
    v = net.visual
    if hasattr(v, 'trunk') and hasattr(v.trunk, 'blocks'):
        return list(v.trunk.blocks)
    if hasattr(v, 'transformer') and hasattr(v.transformer, 'resblocks'):
        return list(v.transformer.resblocks)
    return None

def set_trainable(net, n_blocks):
    for p in net.parameters():
        p.requires_grad = False
    blocks = visual_blocks(net)
    if blocks:
        print(f'found {len(blocks)} visual blocks; unfreezing last {n_blocks}')
        for blk in blocks[-n_blocks:]:
            for p in blk.parameters():
                p.requires_grad = True
    else:
        print('WARN: no blocks found; unfreezing whole visual tower')
        for p in net.visual.parameters():
            p.requires_grad = True
    for name in ('ln_post', 'norm', 'proj', 'head', 'attn_pool', 'trunk'):
        mod = getattr(net.visual, name, None)
        if name == 'trunk' and mod is not None:
            for nm in ('norm', 'head', 'attn_pool'):
                m2 = getattr(mod, nm, None)
                if isinstance(m2, torch.nn.Module):
                    for p in m2.parameters():
                        p.requires_grad = True
        elif isinstance(mod, torch.nn.Module):
            for p in mod.parameters():
                p.requires_grad = True
        elif isinstance(mod, torch.nn.Parameter):
            mod.requires_grad = True
    n = sum(p.numel() for p in net.parameters() if p.requires_grad)
    print(f'trainable encoder params: {n:,}')
    return n

def build_targets(arm):
    keys = sorted({r.designer for r in records})
    idx = {k: i for i, k in enumerate(keys)}
    if arm == 'labels':
        t = torch.tensor([idx[r.designer] for r in records], dtype=torch.long)
        return t, len(keys), 'ce'
    vocab = build_concept_vocab(records, min_designers=2)
    vidx = {c: i for i, c in enumerate(vocab)}
    M = np.zeros((len(records), len(vocab)), dtype=np.float32)
    for i, r in enumerate(records):
        for c in r.concepts:
            if c in vidx:
                M[i, vidx[c]] = 1.0
    print(f'concept targets: {len(vocab)} concepts')
    return torch.tensor(M), len(vocab), 'bce'

def cls_loss(logits, tgt, kind):
    return F.cross_entropy(logits, tgt) if kind == 'ce' \
        else F.binary_cross_entropy_with_logits(logits, tgt)

def lpft(arm):
    model.load_state_dict(INIT_STATE)
    D = base_emb.shape[1]
    targets, out_dim, kind = build_targets(arm)
    targets = targets.to(device)
    fit_t = torch.as_tensor(fit_rows, dtype=torch.long)
    base_feat = torch.tensor(base_emb, dtype=torch.float32, device=device)

    # --- Phase 1: linear probe on frozen features ---
    head = nn.Linear(D, out_dim).to(device)
    optP = torch.optim.Adam(head.parameters(), lr=PROBE_LR, weight_decay=WEIGHT_DECAY)
    Zfit = base_feat[fit_t.to(device)]
    Tfit = targets[fit_t.to(device)]
    for e in range(PROBE_EPOCHS):
        optP.zero_grad()
        loss = cls_loss(head(Zfit), Tfit, kind)
        loss.backward(); optP.step()
    print(f'[{arm}] probe done (final head loss={float(loss):.4f})')

    # --- Phase 2: anchored encoder fine-tune ---
    set_trainable(model, UNFREEZE_BLOCKS)
    for p in head.parameters():
        p.requires_grad = True
    params = [p for p in model.parameters() if p.requires_grad] + list(head.parameters())
    opt = torch.optim.AdamW(params, lr=FT_LR, weight_decay=WEIGHT_DECAY)
    rng = np.random.default_rng(SEED)
    fit = np.asarray(fit_rows)

    best_val = base['top1']
    best_state = copy.deepcopy(model.state_dict())
    stale = 0
    for epoch in range(FT_EPOCHS):
        model.train(); rng.shuffle(fit)
        tot = anch = 0.0; nb = 0
        for i in range(0, len(fit), BATCH):
            rows = fit[i:i+BATCH]
            if len(rows) < 4:
                continue
            x = torch.stack([preprocess_train(pil_images[r]) for r in rows]).to(device)
            f = model.encode_image(x)
            f = f / f.norm(dim=-1, keepdim=True).clamp(min=1e-8)
            ridx = torch.as_tensor(rows, dtype=torch.long, device=device)
            c = cls_loss(head(f), targets[ridx], kind)
            a = ((f - base_feat[ridx]) ** 2).sum(1).mean()
            loss = c + ANCHOR_LAMBDA * a
            opt.zero_grad(); loss.backward(); opt.step()
            tot += float(c); anch += float(a); nb += 1
        emb = embed_all(model)
        val = designer_topk(emb, designers, fit_rows, val_rows)['top1']
        tst = designer_topk(emb, designers, ref_rows, test_rows)['top1']
        print(f'[{arm}] ep {epoch+1}/{FT_EPOCHS} cls={tot/max(1,nb):.3f} '
              f'anchor={anch/max(1,nb):.4f} val_top1={val:.3f} test_top1={tst:.3f}')
        if val > best_val + 1e-4:
            best_val = val; best_state = copy.deepcopy(model.state_dict()); stale = 0
        else:
            stale += 1
            if stale >= PATIENCE:
                print(f'[{arm}] early stop (best val_top1={best_val:.3f})')
                break
    last_emb = embed_all(model)
    model.load_state_dict(best_state)
    best_emb = embed_all(model)
    return best_emb, last_emb, best_state, best_val

## 8. Run the ablation + comparison table

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
results = {'base': {**{k: base[k] for k in ('top1','top3','top5','n_test')},
                    'ci95': bootstrap_ci(base['hit1'])}}
arms = ['concepts', 'labels'] if SUPERVISION == 'both' else [SUPERVISION]
for arm in arms:
    best_emb, last_emb, state, best_val = lpft(arm)
    res  = designer_topk(best_emb, designers, ref_rows, test_rows)
    last = designer_topk(last_emb, designers, ref_rows, test_rows)
    print(f'[{arm}] best-val test top1={res["top1"]:.3f} | last-epoch test top1={last["top1"]:.3f}')
    results[arm] = {**{k: res[k] for k in ('top1','top3','top5','n_test')},
                    'ci95': bootstrap_ci(res['hit1']), 'last_epoch_top1': last['top1']}
    torch.save(state, f'{OUTPUT_DIR}/encoder_{arm}.pt')
    np.savez(f'{OUTPUT_DIR}/runway_{arm}_emb.npz', embeddings=best_emb)
    print(f'saved encoder_{arm}.pt')

print('\n=== Designer top-k (honest by-collection test split) ===')
print(f"{'condition':<10}{'top1':>7}{'  95% CI':>16}{'top3':>7}{'top5':>7}   n")
for k in [c for c in ('base','labels','concepts') if c in results]:
    r = results[k]; ci = f"[{r['ci95'][0]:.3f},{r['ci95'][1]:.3f}]"
    print(f"{k:<10}{r['top1']:>7.3f}{ci:>16}{r['top3']:>7.3f}{r['top5']:>7.3f}   {r['n_test']}")
if 'concepts' in results and 'labels' in results:
    print(f"\nKG-concept vs label-only \u0394top1 = {results['concepts']['top1']-results['labels']['top1']:+.3f}")
    print(f"KG-concept vs base       \u0394top1 = {results['concepts']['top1']-results['base']['top1']:+.3f}")

import json as _json
with open(f'{OUTPUT_DIR}/results.json', 'w') as fh:
    _json.dump(results, fh, indent=2, default=list)
print('\nsaved', f'{OUTPUT_DIR}/results.json')

## 9. How to read it

Watch the `anchor=` term: it should stay small (the embeddings are being held near base), and
`val_top1` / `test_top1` should now **not** collapse toward 0 like the un-anchored run did.

- **`concepts` (or `labels`) test top1 > base 0.408, CIs separated** → the win. Weights are in
  `phase5_out/encoder_*.pt`; load into `FashionEmbedder` locally, rebuild the runway index, re-run
  `fgraph vision eval-runway` to confirm, then the sharper embedder lifts RunwayLinker / RAG / Path A.
- **still flat at base** → this is now a *thorough*, defensible ceiling result across LP-FT too
  (linear probe + anchored fine-tune): ~1.9k runway images cannot improve the pretrained
  FashionSigLIP for cross-collection designer grounding. Report base 0.411 + the ablation.

Levers before giving up: raise `ANCHOR_LAMBDA` if it still drifts, or `UNFREEZE_BLOCKS`/`FT_LR`
slightly if `anchor` is tiny but nothing moves.